In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
fin_data_master = pd.read_csv('financial_data.csv') 

In [3]:
prolong_master = pd.read_csv('prolongations.csv')

In [4]:
fin_data = fin_data_master.copy()
display(fin_data.head(3))

,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


In [5]:
prolong = prolong_master.copy()
display(prolong.head(3))

,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич


In [6]:
fin_data = fin_data.drop(['Причина дубля'], axis=1)

In [7]:
end_data = fin_data.copy()

for col in end_data.columns[1:17]:
    end_data[col] = end_data[col].apply(lambda x: True if x in ['end', 'стоп'] else False)
    
end_data = end_data.groupby(by='id', as_index=False).sum()

for col in end_data.columns[1:17]:
    end_data[col] = end_data[col].apply(lambda x: True if x == 1 else False)

In [8]:
zero_data = fin_data.copy()

for col in zero_data.columns[1:17]:
    zero_data[col] = zero_data[col].apply(lambda x: True if x == 'в ноль' else False)
    
zero_data = zero_data.groupby(by='id', as_index=False).sum()

for col in zero_data.columns[1:17]:
    zero_data[col] = zero_data[col].apply(lambda x: True if x == 1 else False)

In [9]:
pay_data = fin_data.copy()

for col in pay_data.columns[1:17]:
    pay_data[col] = pay_data[col].apply(lambda x: np.nan if x in ['end', 'стоп', 'в ноль'] else x)
    pay_data[col] = pay_data[col].fillna('0,00')
    pay_data[col] = pay_data[col].apply(lambda x: x[:-3])
    pay_data[col] = pay_data[col].apply(lambda x: x.replace('\xa0', ''))
    pay_data[col] = pay_data[col].apply(lambda x: x.replace(',', ''))
    pay_data[col] = pay_data[col].astype(int)
    
pay_data = pay_data.groupby(by='id', as_index=False).sum()

def manager_back(txt):
    name_list = txt.split(' ')
    if len(name_list) <= 3 :
        return txt
    else:    
        t = len(name_list[0])
        name = f'{name_list[0]} {name_list[1]} {name_list[2]}'
        name = name[:-t]
    return name

pay_data['Account'] = pay_data['Account'].apply(manager_back)

In [10]:
# prolong МОДИФИКАЦИЯ НАЗВАНИЯ МЕСЯЦА
def first_symbol(txt):
    frst = txt[0]
    frst = frst.upper()
    txt = frst + txt[1:]
    return txt

prolong['month'] = prolong['month'].apply(first_symbol)


In [ ]:
# НЕ ЗАПУСКАТЬ, объединил через JOIN очень коряво
general_table = pay_data.join(
    other = prolong,
    how = 'left',
    on = 'id',
    lsuffix = '_l',
    rsuffix = '_r'
)

In [11]:
general_table = pd.merge(
    left = pay_data,
    right = prolong,
    how = 'left',
    left_on = 'id',
    right_on = 'id'
)

general_table = general_table.drop(['AM'], axis=1)

In [12]:
close_month_data = general_table.copy()

for col in close_month_data.columns[1:17] :
    for string in close_month_data.index :
        if close_month_data.loc[string, 'month'] == col:
            close_month_data.at[string, col] = True
        else: 
            close_month_data.at[string, col] = False
            
close_month_data = close_month_data.groupby(by='id', as_index=False).sum()

for col in close_month_data.columns[1:17]:
    close_month_data[col] = close_month_data[col].apply(lambda x: True if x == 1 else False)
    
#close_month_data = close_month_data.drop(['Account', 'month'], axis=1)

C:\Users\azott\AppData\Local\Temp\ipykernel_12560\885197620.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  close_month_data.at[string, col] = False
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\885197620.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  close_month_data.at[string, col] = True
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\885197620.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  close_month_data.at[string, col] = False
C:\Use

ПЕРВАЯ ВЕРСИЯ ДО АЛЬТЕРНАТИВЫ, НЕ УВЕРЕН, ЧТО НАДО БЫЛО ДЕЛАТЬ ДЖЕНЕРАЛ

In [265]:
# 1-Я ПРОЛОНГАЦ ИЗ ДЖЕНЕРАЛ
dict_months = {}
for n in range(1, 17):
    dict_months[n] = general_table.columns[n]

first_prolongation = general_table.copy()

first_prolongation['Ноябрь 2022'] = first_prolongation['Ноябрь 2022'].apply(lambda x: 0)

for col in range(2, 17) :
    for string in first_prolongation.index :
        if close_month_data.iloc[string, (col - 1)] == False:
            first_prolongation.at[string, dict_months[col]] = 0

In [266]:
# 2-Я ПРОЛОНГАЦ ИЗ ДЖЕНЕРАЛ
second_prolongation = general_table.copy()

second_prolongation['Ноябрь 2022'] = second_prolongation['Ноябрь 2022'].apply(lambda x: 0)
second_prolongation['Декабрь 2022'] = second_prolongation['Декабрь 2022'].apply(lambda x: 0)

for col in range(3, 17) :
    for string in second_prolongation.index :
        if close_month_data.iloc[string, (col - 2)] == False:
            second_prolongation.at[string, dict_months[col]] = 0

In [ ]:
# ЛАСТ-МОНФС ИЗ ДЖЕНЕРАЛ
last_months_data = general_table.copy()

end_progect_list = []
# идём по каждому месяцу в числовом формате
for col in range(1, 17) :
    # внутри каждого месяца идём по каждому каждому индексу, и заодно сразу находим id проекта
    for string in last_months_data.index :
        id_project = last_months_data.loc[string, 'id']
        # по id проекта находим строку с информацией о последних месяцах этого проекта
        end_index = end_data[end_data['id'] == id_project].index
        if (end_data.iloc[end_index, col]).bool() == True :
            end_progect_list.append(id_project)
            last_months_data.iat[string, col] = 0
            continue
        close_month_index = close_month_data[close_month_data['id'] == id_project].index
        if (close_month_data.iloc[close_month_index, col]).bool() == False :
            last_months_data.iat[string, col] = 0
        else:
            zero_index = zero_data[zero_data['id'] == id_project].index
            if (zero_data.iloc[zero_index, col]).bool() == True :
                pay_index = pay_data[pay_data['id'] == id_project].index
                last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]

C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2853228903.py:11: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (end_data.iloc[end_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2853228903.py:16: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (close_month_data.iloc[close_month_index, col]).bool() == False :
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2853228903.py:20: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (zero_data.iloc[zero_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2853228903.py:22: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]


АЛЬТЕРНАТИВА

In [13]:
# 1-Я ПРОЛОНГАЦ ИЗ ПЭЙ-ДАТА
dict_months = {}
for n in range(1, 17):
    dict_months[n] = pay_data.columns[n]

first_prolongation = pay_data.copy()

first_prolongation['Ноябрь 2022'] = first_prolongation['Ноябрь 2022'].apply(lambda x: 0)

for col in range(2, 17) :
    for string in first_prolongation.index :
        if close_month_data.iloc[string, (col - 1)] == False:
            first_prolongation.at[string, dict_months[col]] = 0

# для подсчёта количества пролонгированных проектов в 1-ый месяц создадим таблицу
first_cnt_prolong = first_prolongation.copy()
            
first_prolongation = first_prolongation.groupby(by='Account', as_index=False).sum()

first_prolongation = first_prolongation.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)
#first_prolongation = first_prolongation.drop([0], axis=0)

In [14]:
# 1-ОЕ КОЛ-ВО ПРОЕКТОВ К ПРОЛОНГАЦИИ
first_cnt_project = close_month_data.copy()

end_project_list = []
for col in range(1, 17):
    for string in first_cnt_project.index :
        id_project = first_cnt_project.loc[string, 'id']
        if id_project in end_project_list :
            first_cnt_project.iat[string, col] = False
        else:
            if end_data.iloc[string, col] == True :
                end_project_list.append(id_project)
                first_cnt_project.iloc[string, col] = False

first_cnt_project['Account'] = first_cnt_project['Account'].apply(manager_back)

first_cnt_project = first_cnt_project.drop(['id', 'month'], axis=1)

first_cnt_project = first_cnt_project.groupby(by='Account', as_index=False).sum()

first_cnt_project_copy = first_cnt_project.copy()
last_month = 'Декабрь 2022'
for month in first_cnt_project.columns[3:14] :
    first_cnt_project[month] = first_cnt_project_copy[last_month]
    last_month = month
    
first_cnt_project = first_cnt_project.drop(['Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

In [15]:
# 1-ОЕ КОЛ-ВО ПРОЛОНГИРОВАННЫХ
# first_cnt_prolong из ячейки [# 1-Я ПРОЛОНГАЦ ИЗ ПЭЙ-ДАТА]

for col in first_cnt_prolong.columns[3:15] :
    first_cnt_prolong[col] = first_cnt_prolong[col].apply(lambda x: False if x == 0 else True)

first_cnt_prolong['Account'] = first_cnt_prolong['Account'].apply(manager_back)

first_cnt_prolong = first_cnt_prolong.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

first_cnt_prolong = first_cnt_prolong.groupby(by='Account', as_index=False).sum()

In [16]:
# 2-Я ПРОЛОНГАЦ ИЗ ПЭЙ-ДАТА
second_prolongation = pay_data.copy()

second_prolongation['Ноябрь 2022'] = second_prolongation['Ноябрь 2022'].apply(lambda x: 0)
second_prolongation['Декабрь 2022'] = second_prolongation['Декабрь 2022'].apply(lambda x: 0)

for col in range(3, 17) :
    for string in second_prolongation.index :
        if close_month_data.iloc[string, (col - 2)] == False:
            second_prolongation.at[string, dict_months[col]] = 0
            
# для подсчёта количества пролонгированных проектов во 2-ой месяц создадим таблицу
second_cnt_prolong = second_prolongation.copy()

second_prolongation = second_prolongation.groupby(by='Account', as_index=False).sum()

second_prolongation = second_prolongation.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)
second_prolongation = second_prolongation.drop([0], axis=0)

In [17]:
# 2-ОЕ КОЛ-ВО ПРОЕКТОВ К ПРОЛОНГАЦИИ
second_cnt_project = close_month_data.copy()

end_project_list = []
for col in range(1, 17):
    for string in second_cnt_project.index :
        id_project = second_cnt_project.loc[string, 'id']
        if id_project in end_project_list :
            second_cnt_project.iat[string, col] = False
        else:
            if end_data.iloc[string, col] == True :
                end_project_list.append(id_project)
                second_cnt_project.iloc[string, col] = False

second_cnt_project['Account'] = second_cnt_project['Account'].apply(manager_back)

second_cnt_project = second_cnt_project.drop(['id', 'month'], axis=1)

second_cnt_project = second_cnt_project.groupby(by='Account', as_index=False).sum()

second_cnt_project_copy = second_cnt_project.copy()
double_last_month = 'Ноябрь 2022'
last_month = 'Декабрь 2022'
for month in second_cnt_project.columns[3:14] :
    second_cnt_project[month] = second_cnt_project_copy[double_last_month]
    double_last_month = last_month
    last_month = month
    
second_cnt_project = second_cnt_project.drop(['Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

In [18]:
# 2-ОЕ КОЛ-ВО ПРОЛОНГИРОВАННЫХ
# second_cnt_prolong из ячейки [# 2-Я ПРОЛОНГАЦ ИЗ ПЭЙ-ДАТА]

for col in second_cnt_prolong.columns[3:15] :
    second_cnt_prolong[col] = second_cnt_prolong[col].apply(lambda x: False if x == 0 else True)

second_cnt_prolong['Account'] = second_cnt_prolong['Account'].apply(manager_back)

second_cnt_prolong = second_cnt_prolong.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

second_cnt_prolong = second_cnt_prolong.groupby(by='Account', as_index=False).sum()

In [ ]:
# ОБЩЕЕ КОЛ-ВО ПРОЕКТОВ К ПРОЛОНГАЦИИ
all_projects_for_first = {}

for month in first_cnt_project.columns[1:13] :
    all_projects_for_first[month] = first_cnt_project[month].sum()
    
all_projects_for_second = {}

for month in second_cnt_project.columns[1:13] :
    all_projects_for_second[month] = second_cnt_project[month].sum()

In [29]:
# ОБЩЕЕ КОЛ-ВО ПРОЛОНГИРОВАННЫХ
all_projects_first_prolong = {}

for month in first_cnt_prolong.columns[1:13] :
    all_projects_first_prolong[month] = first_cnt_prolong[month].sum()
    
all_projects_second_prolong = {}

for month in second_cnt_prolong.columns[1:13] :
    all_projects_second_prolong[month] = second_cnt_prolong[month].sum()

In [19]:
# ЛАСТ-МОНФС ИЗ ПЭЙ-ДАТА
last_months_data = pay_data.copy()

end_project_list = []
# идём по каждому месяцу в числовом формате
for col in range(1, 17) :
    # внутри каждого месяца идём по каждому каждому индексу, и заодно сразу находим id проекта
    for string in last_months_data.index :
        id_project = last_months_data.loc[string, 'id']
        if id_project in end_project_list :
            last_months_data.iat[string, col] = 0
        else:
            # по id проекта находим строку с информацией о последних месяцах этого проекта
            end_index = end_data[end_data['id'] == id_project].index
            if (end_data.iloc[end_index, col]).bool() == True :
                end_project_list.append(id_project)
                last_months_data.iat[string, col] = 0
                continue
            close_month_index = close_month_data[close_month_data['id'] == id_project].index
            if (close_month_data.iloc[close_month_index, col]).bool() == False :
                last_months_data.iat[string, col] = 0
            else:
                zero_index = zero_data[zero_data['id'] == id_project].index
                if (zero_data.iloc[zero_index, col]).bool() == True :
                    pay_index = pay_data[pay_data['id'] == id_project].index
                    last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]

C:\Users\azott\AppData\Local\Temp\ipykernel_12560\2073179320.py:15: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (end_data.iloc[end_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\2073179320.py:20: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (close_month_data.iloc[close_month_index, col]).bool() == False :
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\2073179320.py:24: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (zero_data.iloc[zero_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\2073179320.py:26: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]


In [20]:
# СУММАРНЫЙ КОЭФ ПРОЛОНГАЦИИ
coef_sum_dict = {}

for last_month in range(2, 14) :
    goal_month = last_months_data.columns[last_month + 1]
    coef_sum = last_months_data[last_months_data.columns[last_month]].sum()
    #print(f'целевой месяц {goal_month}, а сумма будет из месяца {last_month} и равна {coef_sum}')
    coef_sum_dict[goal_month] = coef_sum

In [21]:
# 1-ЫЙ КОЭФ ПРОЛОНГАЦИИ
first_coef = first_prolongation.copy()

for col in first_coef.columns[1:13] :
    for string in first_coef.index:
        first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)

C:\Users\azott\AppData\Local\Temp\ipykernel_12560\384861441.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.186' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\384861441.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.257' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\384861441.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.247' has dtype incompatible with int64, p

In [22]:
# 2-ОЙ КОЭФ ПРОЛОНГАЦИИ
second_coef = second_prolongation.copy()

for col in second_coef.columns[1:13] :
    for string in second_coef.index:
        second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict[col] , 3)

C:\Users\azott\AppData\Local\Temp\ipykernel_12560\1738367847.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.012' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\1738367847.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.03' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\1738367847.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.101' has dtype incompatible with in

In [ ]:
# НЕ ЗАПУСКАТЬ, проверил на пропуски
for elem in general_table['month']:
    if elem is np.nan:
        print('есть')

есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
